In [1]:
"""
Lab 6: Mô phỏng cách LLM hoạt động từ mô hình thống kê đến OpenAI API
Chương 6: Đằng sau hậu trường – Cách ChatGPT, GitHub Copilot và các LLM hoạt động
"""
import random
from collections import defaultdict
from typing import Dict, List


# ============================================================
# PHẦN 1: Mô hình xác suất có điều kiện (N-gram)
# ============================================================

def build_ngram_model(text: str, n: int = 1) -> Dict:
    """
    Xây dựng mô hình N-gram để dự đoán ký tự tiếp theo.
    
    Args:
        text: văn bản huấn luyện
        n: số ký tự context
    Returns:
        dict ánh xạ context → {next_char: probability}
    """
    model: Dict = defaultdict(lambda: defaultdict(int))
    text = text.lower()
    
    for i in range(len(text) - n):
        context = text[i:i + n]
        next_char = text[i + n]
        model[context][next_char] += 1
    
    # Chuẩn hóa thành xác suất
    for context in model:
        total = sum(model[context].values())
        model[context] = {c: cnt / total for c, cnt in model[context].items()}
    
    return dict(model)


def generate_text(model: Dict, seed: str, length: int = 100) -> str:
    """
    Sinh văn bản dựa trên mô hình N-gram.
    
    Args:
        model: mô hình N-gram đã xây dựng
        seed: chuỗi khởi đầu
        length: số ký tự cần sinh
    Returns:
        văn bản được sinh ra
    """
    n = len(list(model.keys())[0]) if model else 1
    result = seed.lower()
    
    for _ in range(length):
        context = result[-n:]
        if context not in model:
            break
        
        chars = list(model[context].keys())
        probs = list(model[context].values())
        next_char = random.choices(chars, weights=probs, k=1)[0]
        result += next_char
    
    return result


# ============================================================
# PHẦN 2: Gọi Gemini API (Google Generative AI)
# ============================================================

def call_gemini_api(prompt: str, api_key: str) -> str:
    """
    Gọi Gemini API để sinh code hoặc văn bản.
    
    Args:
        prompt: câu lệnh gửi cho model
        api_key: API key cho Gemini
    Returns:
        phản hồi từ model
    
    Note:
        Cần cài: pip install google-generativeai
    """
    try:
        import google.generativeai as genai
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel('gemini-2.5-flash')
        response = model.generate_content(prompt)
        return response.text
    except ImportError:
        return "[Cần cài: pip install google-generativeai]"
    except Exception as e:
        return f"[Lỗi API: {e}]"


# ============================================================
# DEMO CHẠY
# ============================================================

if __name__ == "__main__":
    print("=" * 60)
    print("LAB 6: Mô Phỏng N-gram Language Model")
    print("=" * 60)
    
    # Dữ liệu ví dụ
    sample_text = """
    The quick brown fox jumps over the lazy dog.
    Python is a programming language that is widely used.
    Machine learning models learn from data patterns.
    Natural language processing enables text understanding.
    """ * 5  # Lặp để có đủ data
    
    print("\n--- Mô hình 1-gram (chỉ nhìn 1 ký tự trước) ---")
    model_1 = build_ngram_model(sample_text, n=1)
    result_1 = generate_text(model_1, seed="the", length=50)
    print(f"Output: {result_1}")
    
    print("\n--- Mô hình 2-gram (nhìn 2 ký tự trước) ---")
    model_2 = build_ngram_model(sample_text, n=2)
    result_2 = generate_text(model_2, seed="th", length=50)
    print(f"Output: {result_2}")
    
    print("\n--- Mô hình 3-gram (nhìn 3 ký tự trước) ---")
    model_3 = build_ngram_model(sample_text, n=3)
    result_3 = generate_text(model_3, seed="the", length=80)
    print(f"Output: {result_3}")
    
    print("\n--- Nhận xét ---")
    print("✓ N càng lớn → output càng giống ngôn ngữ thực")
    print("✓ Nhưng N lớn cần nhiều dữ liệu hơn để có accuracy tốt")
    print("✓ LLM hiện đại dùng Transformer với context window hàng nghìn tokens")
    
    print("\n--- Gọi Gemini API ---")
    api_key = input("Nhập Gemini API key: ")
    result = call_gemini_api("Giải thích Transformer architecture trong 3 câu.", api_key)
    print(f"API Response: {result}")


LAB 6: Mô Phỏng N-gram Language Model

--- Mô hình 1-gram (chỉ nhìn 1 ký tự trước) ---
Output: the    brnage pythels  dogur   
 prom talan odins   w

--- Mô hình 2-gram (nhìn 2 ký tự trước) ---
Output: the lang enables thon is terns.
  
        pythe laz

--- Mô hình 3-gram (nhìn 3 ký tự trước) ---
Output: the quick brown from data patterns.
   python is a programming.
        natural lan

--- Nhận xét ---
✓ N càng lớn → output càng giống ngôn ngữ thực
✓ Nhưng N lớn cần nhiều dữ liệu hơn để có accuracy tốt
✓ LLM hiện đại dùng Transformer với context window hàng nghìn tokens

--- Gọi Gemini API ---


C:\Users\minht\AppData\Local\Temp\ipykernel_16296\1988482767.py:85: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


API Response: Transformer là một kiến trúc mạng nơ-ron sâu cách mạng hóa việc xử lý dữ liệu tuần tự (như ngôn ngữ) bằng cách sử dụng cơ chế Self-Attention. Cơ chế này cho phép mô hình gán trọng số cho các phần khác nhau của chuỗi đầu vào, giúp nắm bắt mối quan hệ ngữ cảnh và xử lý toàn bộ chuỗi song song. Kết hợp với Positional Encoding để bảo toàn thứ tự từ, kiến trúc này cho phép Transformer hiểu ngữ cảnh sâu rộng và xử lý các phụ thuộc xa trong chuỗi một cách hiệu quả, vượt trội so với các mô hình tuần tự truyền thống.
